# 📊 Exploratory Data Analysis — UCI Student Performance Dataset
**Project:** AcademicAI — Student Performance Predictor  
**Author:** Mohammad Umar — B.Tech CSE, COER University  
**Dataset:** UCI Student Performance (Mathematics) — student-mat.csv  
**Records:** 395 students | **Features:** 33 attributes

---
### Sections
1. Import Libraries & Load Data
2. Dataset Overview
3. Target Variable Analysis
4. Numerical Feature Analysis
5. Categorical Feature Analysis
6. Correlation Analysis
7. Feature vs Target Analysis
8. Missing Values & Outliers
9. Key Insights Summary

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Set2')

# Load dataset
df = pd.read_csv('../data/student-mat.csv', sep=';')

# Create target variable
df['target'] = df['G3'].apply(lambda g: 'At-Risk' if g == 0 else ('Fail' if g < 10 else 'Pass'))

print(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

## 2. Dataset Overview

In [ ]:
print('=' * 55)
print('DATASET SHAPE')
print('=' * 55)
print(f'Rows    : {df.shape[0]}')
print(f'Columns : {df.shape[1]}')

print('\n' + '=' * 55)
print('DATA TYPES')
print('=' * 55)
print(df.dtypes.value_counts())

print('\n' + '=' * 55)
print('MISSING VALUES')
print('=' * 55)
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found ✅')

print('\n' + '=' * 55)
print('DUPLICATE ROWS')
print('=' * 55)
print(f'Duplicates: {df.duplicated().sum()}')

In [ ]:
print('STATISTICAL SUMMARY — Numerical Features')
df.describe().round(2)

In [ ]:
print('STATISTICAL SUMMARY — Categorical Features')
df.describe(include='object')

## 3. Target Variable Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Target Variable Analysis', fontsize=15, fontweight='bold', y=1.01)

# G3 distribution
axes[0].hist(df['G3'], bins=20, color='steelblue', edgecolor='white', linewidth=0.8)
axes[0].set_title('G3 Final Grade Distribution')
axes[0].set_xlabel('Final Grade (0–20)')
axes[0].set_ylabel('Number of Students')
axes[0].axvline(10, color='red', linestyle='--', label='Pass threshold (10)')
axes[0].legend()

# Class distribution — bar
class_counts = df['target'].value_counts()
colors = ['#2ecc71', '#e74c3c', '#f39c12']
bars = axes[1].bar(class_counts.index, class_counts.values, color=colors, edgecolor='white')
axes[1].set_title('Class Distribution (Pass / Fail / At-Risk)')
axes[1].set_ylabel('Number of Students')
for bar, val in zip(bars, class_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'{val}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=10)

# Class distribution — pie
axes[2].pie(class_counts.values, labels=class_counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90, pctdistance=0.75,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[2].set_title('Class Proportion')

plt.tight_layout()
plt.show()

print('\nClass Distribution:')
print(df['target'].value_counts().to_string())
print(f'\nClass Imbalance Ratio (Pass:Fail:At-Risk) = '
      f"{class_counts['Pass']}:{class_counts['Fail']}:{class_counts['At-Risk']}")

In [ ]:
# G1, G2, G3 progression
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overlapping distributions
for grade, color, label in [('G1','#3498db','Sem 1'), ('G2','#e67e22','Sem 2'), ('G3','#2ecc71','Final')]:
    axes[0].hist(df[grade], bins=20, alpha=0.5, color=color, label=label, edgecolor='white')
axes[0].set_title('Grade Progression: G1 → G2 → G3')
axes[0].set_xlabel('Grade (0–20)')
axes[0].set_ylabel('Students')
axes[0].axvline(10, color='red', linestyle='--', alpha=0.7, label='Pass line')
axes[0].legend()

# Mean grades per class
grade_means = df.groupby('target')[['G1','G2','G3']].mean()
grade_means.plot(kind='bar', ax=axes[1], color=['#3498db','#e67e22','#2ecc71'],
                 edgecolor='white', rot=0)
axes[1].set_title('Average Grades G1/G2/G3 by Class')
axes[1].set_xlabel('Predicted Class')
axes[1].set_ylabel('Average Grade')
axes[1].legend(['G1','G2','G3'])
axes[1].axhline(10, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 4. Numerical Feature Analysis

In [ ]:
num_cols = ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures',
            'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']

fig, axes = plt.subplots(4, 4, figsize=(18, 14))
fig.suptitle('Numerical Feature Distributions', fontsize=15, fontweight='bold')
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=15, color='steelblue', edgecolor='white', linewidth=0.7)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel('Count')
    mean_val = df[col].mean()
    axes[i].axvline(mean_val, color='red', linestyle='--', linewidth=1.2, label=f'Mean={mean_val:.1f}')
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots — numerical features by class
key_num = ['age', 'studytime', 'failures', 'absences', 'G1', 'G2', 'health']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('Key Numerical Features vs Target Class', fontsize=14, fontweight='bold')
axes = axes.flatten()

order = ['Pass', 'Fail', 'At-Risk']
palette = {'Pass': '#2ecc71', 'Fail': '#e74c3c', 'At-Risk': '#f39c12'}

for i, col in enumerate(key_num):
    sns.boxplot(data=df, x='target', y=col, order=order, palette=palette,
                ax=axes[i], width=0.5)
    axes[i].set_title(f'{col} by Class', fontweight='bold')
    axes[i].set_xlabel('')

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Categorical Feature Analysis

In [ ]:
cat_cols = ['sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob',
            'reason', 'guardian', 'schoolsup', 'famsup', 'paid',
            'activities', 'nursery', 'higher', 'internet', 'romantic']

fig, axes = plt.subplots(4, 4, figsize=(20, 16))
fig.suptitle('Categorical Feature Distributions', fontsize=15, fontweight='bold')
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    counts = df[col].value_counts()
    bars = axes[i].bar(counts.index, counts.values, color=sns.color_palette('Set2', len(counts)),
                       edgecolor='white')
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel('Count')
    for bar, val in zip(bars, counts.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     str(val), ha='center', fontsize=9)
    axes[i].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# Stacked bar — class breakdown per key categorical feature
key_cats = ['sex', 'address', 'Mjob', 'Fjob', 'higher', 'internet', 'failures']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Class Distribution across Key Categorical Features', fontsize=14, fontweight='bold')
axes = axes.flatten()

colors = {'Pass': '#2ecc71', 'Fail': '#e74c3c', 'At-Risk': '#f39c12'}

for i, col in enumerate(key_cats):
    ct = pd.crosstab(df[col], df['target'], normalize='index') * 100
    ct = ct.reindex(columns=['Pass', 'Fail', 'At-Risk'], fill_value=0)
    ct.plot(kind='bar', stacked=True, ax=axes[i],
            color=[colors[c] for c in ct.columns], edgecolor='white', rot=20)
    axes[i].set_title(f'{col} → Class %', fontweight='bold')
    axes[i].set_ylabel('Percentage (%)')
    axes[i].set_xlabel('')
    axes[i].legend(loc='upper right', fontsize=8)
    axes[i].yaxis.set_major_formatter(mticker.PercentFormatter())

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
# Full correlation heatmap — numerical columns
num_df = df[num_cols].copy()
corr = num_df.corr()

plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, linecolor='white',
            annot_kws={'size': 9}, vmin=-1, vmax=1)
plt.title('Correlation Matrix — Numerical Features', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with G3 (most important — final grade)
g3_corr = num_df.corr()['G3'].drop('G3').sort_values(ascending=True)

colors_bar = ['#e74c3c' if v < 0 else '#2ecc71' for v in g3_corr.values]
plt.figure(figsize=(10, 6))
plt.barh(g3_corr.index, g3_corr.values, color=colors_bar, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Correlation of Features with G3 (Final Grade)', fontsize=13, fontweight='bold')
plt.xlabel('Pearson Correlation Coefficient')
for i, (val, name) in enumerate(zip(g3_corr.values, g3_corr.index)):
    plt.text(val + (0.01 if val >= 0 else -0.01), i,
             f'{val:.2f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.show()

print('\nTop 5 positive correlations with G3:')
print(g3_corr.tail(5).to_string())
print('\nTop 5 negative correlations with G3:')
print(g3_corr.head(5).to_string())

## 7. Feature vs Target Analysis

In [ ]:
# G1 vs G2 scatter coloured by class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

palette = {'Pass': '#2ecc71', 'Fail': '#e74c3c', 'At-Risk': '#f39c12'}
for cls, grp in df.groupby('target'):
    axes[0].scatter(grp['G1'], grp['G2'], label=cls, color=palette[cls],
                    alpha=0.6, edgecolors='white', linewidth=0.4, s=50)
axes[0].set_title('G1 vs G2 — coloured by outcome', fontweight='bold')
axes[0].set_xlabel('G1 (1st term grade)')
axes[0].set_ylabel('G2 (2nd term grade)')
axes[0].axhline(10, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(10, color='gray', linestyle='--', alpha=0.5)
axes[0].legend()

# G2 vs G3
for cls, grp in df.groupby('target'):
    axes[1].scatter(grp['G2'], grp['G3'], label=cls, color=palette[cls],
                    alpha=0.6, edgecolors='white', linewidth=0.4, s=50)
axes[1].set_title('G2 vs G3 — coloured by outcome', fontweight='bold')
axes[1].set_xlabel('G2 (2nd term grade)')
axes[1].set_ylabel('G3 (final grade)')
axes[1].axhline(10, color='gray', linestyle='--', alpha=0.5)
axes[1].axvline(10, color='gray', linestyle='--', alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Absences vs G3
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cls, grp in df.groupby('target'):
    axes[0].scatter(grp['absences'], grp['G3'], label=cls, color=palette[cls],
                    alpha=0.6, edgecolors='white', s=50)
axes[0].set_title('Absences vs G3 Final Grade', fontweight='bold')
axes[0].set_xlabel('Number of Absences')
axes[0].set_ylabel('Final Grade G3')
axes[0].axhline(10, color='gray', linestyle='--', alpha=0.5)
axes[0].legend()

# Study time vs G3 violin
sns.violinplot(data=df, x='studytime', y='G3', hue='target',
               palette=palette, ax=axes[1], split=False, inner='quartile')
axes[1].set_title('Study Time vs G3 by Class', fontweight='bold')
axes[1].set_xlabel('Study Time (1=<2h, 2=2-5h, 3=5-10h, 4=>10h)')
axes[1].set_ylabel('Final Grade G3')
axes[1].axhline(10, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Failures analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fail_class = pd.crosstab(df['failures'], df['target'], normalize='index') * 100
fail_class = fail_class.reindex(columns=['Pass', 'Fail', 'At-Risk'], fill_value=0)
fail_class.plot(kind='bar', ax=axes[0],
                color=[palette[c] for c in fail_class.columns],
                edgecolor='white', rot=0)
axes[0].set_title('Past Failures → Class Percentage', fontweight='bold')
axes[0].set_xlabel('Number of Past Failures')
axes[0].set_ylabel('Percentage (%)')
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter())

# Mother's education vs G3
medu_g3 = df.groupby('Medu')['G3'].mean()
axes[1].bar(medu_g3.index, medu_g3.values, color='steelblue', edgecolor='white')
axes[1].set_title("Mother's Education Level vs Avg G3", fontweight='bold')
axes[1].set_xlabel('Medu (0=none → 4=higher)')
axes[1].set_ylabel('Average Final Grade G3')
axes[1].axhline(10, color='red', linestyle='--', alpha=0.6, label='Pass line')
axes[1].legend()
for i, (x, y) in enumerate(zip(medu_g3.index, medu_g3.values)):
    axes[1].text(x, y + 0.2, f'{y:.1f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 8. Missing Values & Outliers

In [ ]:
# Missing values heatmap
plt.figure(figsize=(16, 3))
sns.heatmap(df.isnull().T, cbar=False, cmap='Reds', yticklabels=True)
plt.title('Missing Values Heatmap (Red = Missing)', fontweight='bold')
plt.tight_layout()
plt.show()

total_missing = df.isnull().sum().sum()
print(f'Total missing values: {total_missing}')
if total_missing == 0:
    print('✅ No missing values — dataset is complete!')

In [ ]:
# Outlier detection using IQR
outlier_cols = ['age', 'absences', 'G1', 'G2', 'G3', 'studytime', 'failures']

fig, axes = plt.subplots(1, len(outlier_cols), figsize=(18, 5))
fig.suptitle('Outlier Detection — Box Plots', fontsize=13, fontweight='bold')

for i, col in enumerate(outlier_cols):
    axes[i].boxplot(df[col], patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6),
                    medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(col, fontweight='bold')
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    axes[i].set_xlabel(f'{len(outliers)} outliers', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

print('Outlier Count per Feature (IQR method):')
for col in outlier_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    n = len(df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)])
    print(f'  {col:12s}: {n} outliers ({n/len(df)*100:.1f}%)')

## 9. Key Insights Summary

In [ ]:
print('=' * 60)
print('KEY INSIGHTS — AcademicAI EDA')
print('=' * 60)

print(f"""
📦 DATASET
  • {len(df)} students, {df.shape[1]} features, 0 missing values
  • Semicolon-delimited CSV (UCI Portugal Math dataset)

🎯 TARGET VARIABLE (G3 → Pass/Fail/At-Risk)
  • Pass    (G3 ≥ 10) : {(df['target']=='Pass').sum()} students ({(df['target']=='Pass').mean()*100:.1f}%)
  • Fail    (G3 1–9)  : {(df['target']=='Fail').sum()} students ({(df['target']=='Fail').mean()*100:.1f}%)
  • At-Risk (G3 = 0)  : {(df['target']=='At-Risk').sum()} students ({(df['target']=='At-Risk').mean()*100:.1f}%)
  ⚠ Class imbalance present — 'balanced' class weighting applied in model

📈 GRADE ANALYSIS
  • G1 mean: {df['G1'].mean():.1f} | G2 mean: {df['G2'].mean():.1f} | G3 mean: {df['G3'].mean():.1f}
  • G1–G3 correlation: {df['G1'].corr(df['G3']):.3f}
  • G2–G3 correlation: {df['G2'].corr(df['G3']):.3f}  ← strongest predictor
  • Students who scored 0 in G2 almost always score 0 in G3

🏫 ATTENDANCE
  • Mean absences: {df['absences'].mean():.1f} | Max: {df['absences'].max()}
  • Absences–G3 correlation: {df['absences'].corr(df['G3']):.3f} (negative — more absences = lower grade)
  • At-Risk students have on average {df[df['target']=='At-Risk']['absences'].mean():.1f} absences
    vs {df[df['target']=='Pass']['absences'].mean():.1f} for Pass students

📚 STUDY TIME
  • Most students study 2–5 hrs/week (band 2)
  • Pass students avg study time: {df[df['target']=='Pass']['studytime'].mean():.2f}
  • Fail students avg study time: {df[df['target']=='Fail']['studytime'].mean():.2f}

❌ PAST FAILURES
  • 0 failures: {(df['failures']==0).sum()} students ({(df['failures']==0).mean()*100:.1f}%)
  • Students with 3+ failures have {(df[df['failures']>=3]['target']=='Pass').mean()*100:.1f}% pass rate
  • Students with 0 failures have {(df[df['failures']==0]['target']=='Pass').mean()*100:.1f}% pass rate

👨‍👩‍👦 FAMILY BACKGROUND
  • Mother's education (Medu) positively correlated with G3: {df['Medu'].corr(df['G3']):.3f}
  • Students whose mother has higher education avg G3: {df[df['Medu']==4]['G3'].mean():.1f}
  • Students whose mother has no education avg G3: {df[df['Medu']==0]['G3'].mean():.1f}

✅ MODEL IMPLICATIONS
  • G2 is the strongest single predictor (corr={df['G2'].corr(df['G3']):.3f})
  • Absences and past failures are the top non-grade predictors
  • At-Risk class is heavily underrepresented — requires class weighting
  • No missing values — no imputation needed
""")